# SWaT — Unsupervised Anomaly Detection (Generalizable Pipeline)

**Goal:** train several anomaly-detection models on *normal* behavior only, test them on attack data, and compare them in one notebook on a **GPU (RTX 5000)**. The same pipeline runs on a second dataset (**WADI**) to demonstrate generalization.

**Models (unsupervised, trained on normal only):**
1. **Isolation Forest** — classical baseline (CPU, sklearn).
2. **Dense Autoencoder (AE)** — reconstructs a normal timestep (PyTorch, GPU).
3. **LSTM Autoencoder (LSTM-AE)** — captures temporal patterns over a window (PyTorch, GPU).
4. **USAD** — one encoder, two decoders trained adversarially (PyTorch, GPU).
5. **TranAD** — transformer-based, adversarial two-decoder; strong on SWaT (PyTorch, GPU).
6. **GDN** — Graph Deviation Network; learns inter-sensor relations as a graph, forecasts each sensor; strong on WADI (PyTorch, GPU).
7. **Ensemble** — z-scored average of the best window models (LSTM-AE + TranAD + GDN); often beats any single model.

**Generalization idea:** every step is a reusable function. The whole per-dataset flow is wrapped in `run_all_models(...)`, so running on another dataset (e.g. WADI) is a single call — the architecture transfers, the weights are retrained. See the **Generalization** section, which runs SWaT and WADI and prints one combined comparison table.

> **How to run:** execute cells top to bottom (Run All). The `CONFIG` cell holds every setting — edit it, then Run All.


## 0. Running the notebook in VS Code (setup)

One-time steps:

1. Install these VS Code extensions (from Microsoft): `Python` and `Jupyter`.
2. Create and activate a virtual environment:

```bash
python -m venv .venv
# Windows:
.venv\Scripts\activate
# Linux/Mac:
source .venv/bin/activate
```

3. Install the libraries and a GPU build of PyTorch (CUDA 12.1 example — match your RTX 5000 driver):

```bash
pip install torch --index-url https://download.pytorch.org/whl/cu121
pip install numpy pandas scikit-learn matplotlib tqdm openpyxl joblib
```

4. Open the `.ipynb` file, and at the top-right select the **Kernel** from `.venv`.
5. Confirm the GPU is detected in the next cell (it should print `cuda`).

> If `torch.cuda.is_available()` returns `False`, the PyTorch CUDA build does not match your driver — reinstall with the correct CUDA version.


## 1. Imports & GPU check

In [ ]:
import os, glob, time, json, warnings, random, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (precision_recall_fscore_support, roc_auc_score,
                             average_precision_score)
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device :", DEVICE)
if DEVICE.type == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))
    print("Memory :", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")
else:
    print("WARNING: GPU not available - running on CPU (slow). See section 0.")

## 2. CONFIG — edit here only

All control lives in this cell. `DATASETS` maps a name to its folder; add or change paths to run on different data.

In [ ]:
CONFIG = {
    # --- datasets: name -> folder of CSV files ---
    "DATASETS": {
        "SWaT": "./SWaT dataset",
        "WADI": "./data/wadi",     # used in the Generalization section (skipped if missing)
    },
    # exclude files starting with m/M so a merged.csv isn't loaded on top of the split files
    "FILE_GLOB": "[!mM]*.csv",

    # --- columns (auto-detected if None) ---
    "LABEL_COL": None,             # e.g. "Normal/Attack" - leave None for auto-detect
    "TIME_COL":  None,             # timestamp column if present - dropped

    # --- preprocessing ---
    "SUBSAMPLE": 5,                # keep 1 row every N (SWaT is 1 Hz; 5 = 5x faster). 1 = all rows
    "WINDOW": 30,                  # temporal window length for sequence models (after subsample)
    "STRIDE": 1,
    "VAL_FRACTION": 0.2,           # fraction of normal used for validation / threshold

    # --- feature selection ---
    "VAR_THRESHOLD": 1e-8,         # drop near-constant columns (std <= this)
    "MAX_FEATURES": None,          # keep only top-K highest-variance features (None = keep all)

    # --- per-dataset overrides (merged into cfg for that dataset only) ---
    "DATASET_OVERRIDES": {
        # WADI: longer window (slower attacks) + stronger feature selection (keep 50 best of ~98)
        "WADI": {"WINDOW": 60, "MAX_FEATURES": 50},
    },

    # --- optional time-range labels (build labels from the time column) ---
    # If a dataset name appears here, every row whose timestamp falls in any [start, end]
    # range is labeled attack. Use this when the CSV has no usable label column (WADI's
    # attacks come from a separate schedule). Fill with the official WADI attack windows.
    "ATTACK_WINDOWS": {
        # "WADI": [("2017-10-09 19:25:00", "2017-10-09 19:50:16"),
        #          ("2017-10-10 10:24:10", "2017-10-10 10:34:00")],
    },

    # --- training ---
    "EPOCHS": 30,
    "USAD_EPOCHS": 100,            # USAD needs more epochs to converge (was collapsing at 30)
    "TRANAD_EPOCHS": 30,           # TranAD (transformer) - keep moderate; raise if underfitting
    "GDN_EPOCHS": 30,              # GDN (graph) forecasting model
    "GDN_TOPK": 15,                # neighbours per sensor in the learned graph
    "BATCH": 512,
    "LR": 1e-3,
    "THRESHOLD_PERCENTILE": 99.0,  # fixed threshold (kept only as a reference column)
    "THRESHOLD_MODE": "best_f1",   # "best_f1" = search threshold that maximizes F1 (recommended)
    "POINT_ADJUST": True,          # point-adjust protocol (standard in SWaT papers)

    # --- which models to run ---
    "RUN": {"IsolationForest": True, "DenseAE": True, "LSTM_AE": True, "USAD": True,
            "TranAD": True, "GDN": True},

    # ensemble = z-scored average of these window models (present ones only; USAD excluded as weak)
    "ENSEMBLE_MEMBERS": ["LSTM_AE", "TranAD", "GDN"],

    # --- output ---
    "OUT_DIR": "./artifacts",
}
os.makedirs(CONFIG["OUT_DIR"], exist_ok=True)
print(json.dumps(CONFIG, indent=2))

## 3. Flexible CSV loader

Reads every CSV in a folder, strips whitespace from column names (SWaT has leading spaces), and builds labels one of two ways:
- **label column** (auto-detected): any value containing "attack" (incl. the `A ttack` typo) becomes **1**.
- **time ranges** (`CONFIG["ATTACK_WINDOWS"]`): rows whose timestamp falls in an attack window become **1** — use this for WADI, whose CSV has no clean label column.

It also applies **feature selection**: drops near-constant columns (`VAR_THRESHOLD`) and can keep only the top-K highest-variance features (`MAX_FEATURES`).

In [ ]:
def _find_col(cols, candidates):
    low = {c.lower().strip(): c for c in cols}
    for cand in candidates:
        if cand.lower() in low:
            return low[cand.lower()]
    return None

def load_dataset(data_dir, cfg):
    files = sorted(glob.glob(os.path.join(data_dir, cfg["FILE_GLOB"])))
    if not files:
        raise FileNotFoundError(f"No files in {data_dir} - check the path")
    print("Files:", [os.path.basename(f) for f in files])
    frames = []
    for f in files:
        df = pd.read_csv(f, low_memory=False)
        df.columns = [str(c).strip() for c in df.columns]   # strip whitespace from names
        frames.append(df)
    df = pd.concat(frames, ignore_index=True)

    # timestamp column - keep aside (needed for time-range labels and diagnostics)
    time_col = cfg["TIME_COL"] or _find_col(df.columns, ["Timestamp", "time", "date", "row", "datetime"])
    tvals = df[time_col] if (time_col and time_col in df.columns) else None

    # ---- labels ----
    ds_name = cfg.get("_dataset_name")
    windows = cfg.get("ATTACK_WINDOWS", {}).get(ds_name) if ds_name else None
    label_col = None
    if windows and tvals is not None:
        t = pd.to_datetime(tvals, errors="coerce")
        y = pd.Series(0, index=df.index, dtype=int)
        for (s, e) in windows:
            y[(t >= pd.to_datetime(s)) & (t <= pd.to_datetime(e))] = 1
        print(f"Labels from {len(windows)} time ranges (column '{time_col}')")
    else:
        label_col = cfg["LABEL_COL"] or _find_col(
            df.columns, ["Normal/Attack", "label", "y", "attack", "class"])
        if label_col is None:
            raise ValueError(f"No label column found. Columns: {list(df.columns)[-6:]}. "
                             f"Set CONFIG['ATTACK_WINDOWS']['{ds_name}'] to label from time ranges.")
        print("Label column:", label_col)
        raw = df[label_col].astype(str).str.strip().str.lower().str.replace(" ", "")
        y = raw.apply(lambda v: 1 if "attack" in v else (1 if v in {"1", "1.0"} else 0)).astype(int)

    # features = everything except time + label
    drop = [c for c in [time_col, label_col] if c and c in df.columns]
    X = df.drop(columns=drop)

    # numeric + fill gaps
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.dropna(axis=1, how="all").ffill().fillna(0.0)

    # feature selection: drop near-constant columns (std <= VAR_THRESHOLD)
    std = X.std()
    keep = std[std > cfg.get("VAR_THRESHOLD", 0.0)].index.tolist()
    dropped = X.shape[1] - len(keep)
    X = X[keep]
    if dropped:
        print(f"Dropped {dropped} near-constant columns")

    # optional: keep only the top-K highest-variance features
    k = cfg.get("MAX_FEATURES")
    if k and X.shape[1] > k:
        top = X.var().sort_values(ascending=False).head(k).index.tolist()
        X = X[top]
        print(f"Kept top {k} features by variance")

    print(f"Shape: X={X.shape}, attack ratio={y.mean():.3%}")
    return X.reset_index(drop=True), y.reset_index(drop=True)

## 3.5 Label sanity check (run this first for WADI)

Before training, verify each dataset's labels loaded correctly. If **all models score near the attack ratio** (e.g. PR-AUC ~0.06 on WADI), the usual cause is *wrong labels*, not weak models. This cell prints the attack ratio, number of contiguous attack segments, and feature count per dataset, and warns on obviously broken labels.

In [ ]:
def label_sanity_check(cfg):
    for name, d in cfg["DATASETS"].items():
        fs = glob.glob(os.path.join(d, cfg["FILE_GLOB"]))
        if not fs:
            print(f"[{name}] no files at {d} - skipped"); continue
        try:
            X, y = load_dataset(d, {**cfg, "_dataset_name": name})
        except Exception as e:
            print(f"[{name}] load error: {e}"); continue
        yv = y.values
        segs = int(((yv[1:] == 1) & (yv[:-1] == 0)).sum() + (1 if len(yv) and yv[0] == 1 else 0))
        print(f"[{name}] rows={len(yv):>8}  attack_ratio={yv.mean():.3%}  attack_segments={segs}  features={X.shape[1]}")
        if yv.mean() == 0:
            print("   WARNING: no attacks labeled - label column/time-ranges are wrong.")
        elif yv.mean() > 0.9:
            print("   WARNING: almost everything labeled attack - parsing likely inverted.")
        elif segs < 2:
            print("   NOTE: only one attack segment - check this matches the dataset's attack count.")
        print()
    print("If WADI looks wrong, fill CONFIG['ATTACK_WINDOWS']['WADI'] with the official attack schedule (uses the time column).")

label_sanity_check(CONFIG)

## 4. Split, scale, window

**Golden rule:** the scaler is fit on normal data only, so the models never see an attack during training. That is the basis of the unsupervised approach.

In [ ]:
def make_splits(X, y, cfg):
    X = X.iloc[::cfg["SUBSAMPLE"]].reset_index(drop=True)   # subsample for speed
    y = y.iloc[::cfg["SUBSAMPLE"]].reset_index(drop=True)

    normal_idx = y[y == 0].index
    n_val = int(len(normal_idx) * cfg["VAL_FRACTION"])
    train_idx = normal_idx[:-n_val]     # keep time order: early normal = train
    val_idx   = normal_idx[-n_val:]     # later normal = validation

    scaler = StandardScaler().fit(X.iloc[train_idx].values)   # fit on normal only
    Xtr  = scaler.transform(X.iloc[train_idx].values)
    Xval = scaler.transform(X.iloc[val_idx].values)
    Xte  = scaler.transform(X.values)   # test = everything (contains attacks)
    yte  = y.values
    print(f"Train(normal)={Xtr.shape}  Val(normal)={Xval.shape}  Test(all)={Xte.shape}")
    return Xtr, Xval, Xte, yte, scaler

def make_windows(arr, window, stride):
    """(T, F) -> (num_windows, window, F)"""
    if len(arr) < window:
        return np.empty((0, window, arr.shape[1]), dtype=np.float32)
    idx = range(0, len(arr) - window + 1, stride)
    return np.stack([arr[i:i+window] for i in idx]).astype(np.float32)

def window_labels(y, window, stride):
    """window label = 1 if it contains any attack point"""
    idx = range(0, len(y) - window + 1, stride)
    return np.array([1 if y[i:i+window].max() > 0 else 0 for i in idx], dtype=int)

## 5. Evaluation helpers

`point_adjust` is the common SWaT protocol: if the model flags *any* point inside a contiguous attack segment, the whole segment counts as detected.

**Threshold:** a fixed percentile often lands above every test score (→ F1 = 0 even when ROC-AUC is high, which is a *threshold* problem, not a model problem). So the default is `THRESHOLD_MODE = "best_f1"`: we search the threshold that maximizes point-adjusted F1 over the test-score distribution — the standard "best-F1" protocol in SWaT papers. We keep the fixed-percentile F1 as a reference column (`F1_fixed`).

**Metrics:** for balanced data read **F1**; for highly imbalanced data (WADI, ~5% attacks) **PR-AUC** is the more honest headline because it does not depend on a single threshold.

In [ ]:
def point_adjust(y_true, y_pred):
    y_pred = y_pred.copy()
    i = 0; n = len(y_true)
    while i < n:
        if y_true[i] == 1:
            j = i
            while j < n and y_true[j] == 1:
                j += 1
            if y_pred[i:j].any():
                y_pred[i:j] = 1
            i = j
        else:
            i += 1
    return y_pred

def _prf(y_true, y_pred, cfg):
    if cfg["POINT_ADJUST"]:
        y_pred = point_adjust(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    return p, r, f1

def best_f1_threshold(scores, y_true, cfg, n_grid=200):
    """Search the threshold that maximizes (point-adjusted) F1 over the score distribution."""
    grid = np.unique(np.quantile(scores, np.linspace(0.50, 0.9995, n_grid)))
    best = {"f1": -1.0, "thr": grid[-1], "p": 0.0, "r": 0.0}
    for t in grid:
        p, r, f1 = _prf(y_true, (scores > t).astype(int), cfg)
        if f1 > best["f1"]:
            best = {"f1": f1, "thr": float(t), "p": p, "r": r}
    return best

def evaluate(name, dataset, scores_val, scores_test, y_true, cfg):
    # reference: fixed percentile threshold from normal-validation error
    thr_fixed = np.percentile(scores_val, cfg["THRESHOLD_PERCENTILE"])
    _, _, f1_fixed = _prf(y_true, (scores_test > thr_fixed).astype(int), cfg)

    if cfg.get("THRESHOLD_MODE", "best_f1") == "best_f1":
        b = best_f1_threshold(scores_test, y_true, cfg)
        p, r, f1 = b["p"], b["r"], b["f1"]
    else:
        p, r, f1 = _prf(y_true, (scores_test > thr_fixed).astype(int), cfg)

    try:
        auc = roc_auc_score(y_true, scores_test)
        ap  = average_precision_score(y_true, scores_test)
    except ValueError:
        auc = ap = float("nan")
    res = {"Dataset": dataset, "Model": name, "Precision": round(p,4), "Recall": round(r,4),
           "F1": round(f1,4), "F1_fixed": round(f1_fixed,4),
           "ROC_AUC": round(auc,4), "PR_AUC": round(ap,4)}
    print(f"  {name:16s} | F1={f1:.3f}  P={p:.3f}  R={r:.3f}  AUC={auc:.3f}  PR-AUC={ap:.3f}")
    return res

## 6. Model definitions & training utilities

Building blocks only (nothing runs here). All deep models train with the same loop and move data to the GPU.

In [ ]:
# ---------- shared training utilities ----------
def fit_minmax_windows(Wtr):
    """Fit MinMax [0,1] on normal training windows; return a transform for any window set.
    Used by USAD and TranAD, whose decoders end in Sigmoid and need bounded inputs."""
    mm = MinMaxScaler().fit(Wtr.reshape(len(Wtr), -1))
    def tf(Wx):
        if len(Wx) == 0: return Wx
        return mm.transform(Wx.reshape(len(Wx), -1)).reshape(Wx.shape).astype(np.float32)
    return tf

def train_ae(model, Xtr_or_Wtr, cfg):
    """Generic autoencoder training: loss = MSE(x, recon). Accepts flat (N,F) or windows (N,W,F)."""
    model.to(DEVICE)
    loader = DataLoader(TensorDataset(torch.tensor(Xtr_or_Wtr, dtype=torch.float32)),
                        batch_size=cfg["BATCH"], shuffle=True, drop_last=True)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["LR"]); mse = nn.MSELoss()
    for ep in range(cfg["EPOCHS"]):
        model.train(); tot = 0.0
        for (xb,) in loader:
            xb = xb.to(DEVICE)
            opt.zero_grad(); loss = mse(model(xb), xb)
            loss.backward(); opt.step(); tot += loss.item()*xb.size(0)
        if (ep+1) % max(1, cfg["EPOCHS"]//6) == 0:
            print(f"    epoch {ep+1:3d}/{cfg['EPOCHS']}  loss={tot/len(loader.dataset):.5f}")
    return model

@torch.no_grad()
def recon_error(model, X, seq, batch=1024):
    model.eval(); errs = []
    for i in range(0, len(X), batch):
        xb = torch.tensor(X[i:i+batch], dtype=torch.float32, device=DEVICE)
        r = model(xb)
        dims = (1, 2) if seq else 1
        errs.append(((r - xb) ** 2).mean(dim=dims).cpu().numpy())
    return np.concatenate(errs)

# ---------- Dense Autoencoder ----------
class DenseAE(nn.Module):
    def __init__(self, d, latent=16):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(d,64), nn.ReLU(), nn.Linear(64,32),
                                 nn.ReLU(), nn.Linear(32,latent), nn.ReLU())
        self.dec = nn.Sequential(nn.Linear(latent,32), nn.ReLU(), nn.Linear(32,64),
                                 nn.ReLU(), nn.Linear(64,d))
    def forward(self, x): return self.dec(self.enc(x))

# ---------- LSTM Autoencoder ----------
class LSTMAE(nn.Module):
    def __init__(self, d, hidden=64, latent=32):
        super().__init__()
        self.enc = nn.LSTM(d, hidden, batch_first=True)
        self.to_lat = nn.Linear(hidden, latent)
        self.from_lat = nn.Linear(latent, hidden)
        self.dec = nn.LSTM(hidden, hidden, batch_first=True)
        self.out = nn.Linear(hidden, d)
    def forward(self, x):
        B, T, D = x.shape
        _, (h, _) = self.enc(x)
        z = self.to_lat(h[-1])
        dec_in = self.from_lat(z).unsqueeze(1).repeat(1, T, 1)
        dec_out, _ = self.dec(dec_in)
        return self.out(dec_out)

# ---------- USAD (encoder + two decoders, adversarial) ----------
class USAD(nn.Module):
    def __init__(self, in_dim, latent=40):
        super().__init__()
        def enc(): return nn.Sequential(nn.Linear(in_dim, in_dim//2), nn.ReLU(),
                        nn.Linear(in_dim//2, in_dim//4), nn.ReLU(),
                        nn.Linear(in_dim//4, latent), nn.ReLU())
        def dec(): return nn.Sequential(nn.Linear(latent, in_dim//4), nn.ReLU(),
                        nn.Linear(in_dim//4, in_dim//2), nn.ReLU(),
                        nn.Linear(in_dim//2, in_dim), nn.Sigmoid())   # Sigmoid: USAD needs [0,1]-bounded output to stay stable
        self.E, self.D1, self.D2 = enc(), dec(), dec()
    def forward(self, w):
        z = self.E(w)
        return self.D1(z), self.D2(z), self.D2(self.E(self.D1(z)))

def train_usad(model, Wtr, cfg):
    model.to(DEVICE)
    loader = DataLoader(TensorDataset(torch.tensor(Wtr, dtype=torch.float32)),
                        batch_size=cfg["BATCH"], shuffle=True, drop_last=True)
    opt1 = torch.optim.Adam(list(model.E.parameters())+list(model.D1.parameters()), lr=cfg["LR"])
    opt2 = torch.optim.Adam(list(model.E.parameters())+list(model.D2.parameters()), lr=cfg["LR"])
    epochs = cfg.get("USAD_EPOCHS", cfg["EPOCHS"])   # USAD needs more epochs to converge
    for ep in range(1, epochs+1):
        model.train(); t1=t2=0.0; n=ep
        for (wb,) in loader:
            wb = wb.to(DEVICE).view(wb.size(0), -1)      # flatten window
            z = model.E(wb); w1 = model.D1(z)
            w2_from1 = model.D2(model.E(w1))
            loss1 = (1/n)*torch.mean((wb-w1)**2) + (1-1/n)*torch.mean((wb-w2_from1)**2)
            opt1.zero_grad(); loss1.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # stabilize adversarial loss
            opt1.step()
            z = model.E(wb); w2 = model.D2(z); w1 = model.D1(z)
            w2_from1 = model.D2(model.E(w1))
            loss2 = (1/n)*torch.mean((wb-w2)**2) - (1-1/n)*torch.mean((wb-w2_from1)**2)
            opt2.zero_grad(); loss2.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt2.step()
            t1 += loss1.item(); t2 += loss2.item()
        if ep % max(1, epochs//6) == 0:
            print(f"    epoch {ep:3d}/{epochs}  L1={t1/len(loader):.5f}  L2={t2/len(loader):.5f}")
    return model

@torch.no_grad()
def usad_score(model, W, alpha=0.5, beta=0.5, batch=256):
    model.eval(); out=[]
    for i in range(0, len(W), batch):
        wb = torch.tensor(W[i:i+batch], dtype=torch.float32, device=DEVICE)
        wb = wb.reshape(wb.size(0), -1)
        w1, _, w2 = model(wb)
        s = alpha*torch.mean((wb-w1)**2, dim=1) + beta*torch.mean((wb-w2)**2, dim=1)
        out.append(s.cpu().numpy())
    return np.concatenate(out)

# ---------- TranAD (transformer, adversarial two-decoder) — SOTA on WADI ----------
# Faithful compact version of Tuli et al. 2022. Input windows are MinMax-scaled to [0,1]
# (the output FCN ends in Sigmoid). It reconstructs the LAST timestep of each window,
# conditioned on a focus score (phase-1 reconstruction error), then refines adversarially.
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(1))   # (max_len, 1, d_model)
    def forward(self, x):
        return self.dropout(x + self.pe[:x.size(0)])

class TranAD(nn.Module):
    def __init__(self, feats, window):
        super().__init__()
        self.n_feats = feats
        self.pos_encoder = PositionalEncoding(2*feats, window)
        enc = nn.TransformerEncoderLayer(d_model=2*feats, nhead=feats, dim_feedforward=16, dropout=0.1)
        self.transformer_encoder = nn.TransformerEncoder(enc, 1)
        dec1 = nn.TransformerDecoderLayer(d_model=2*feats, nhead=feats, dim_feedforward=16, dropout=0.1)
        self.transformer_decoder1 = nn.TransformerDecoder(dec1, 1)
        dec2 = nn.TransformerDecoderLayer(d_model=2*feats, nhead=feats, dim_feedforward=16, dropout=0.1)
        self.transformer_decoder2 = nn.TransformerDecoder(dec2, 1)
        self.fcn = nn.Sequential(nn.Linear(2*feats, feats), nn.Sigmoid())
    def encode(self, src, c, tgt):
        src = torch.cat((src, c), dim=2) * math.sqrt(self.n_feats)
        src = self.pos_encoder(src)
        memory = self.transformer_encoder(src)
        return tgt.repeat(1, 1, 2), memory
    def forward(self, src, tgt):
        c = torch.zeros_like(src)
        x1 = self.fcn(self.transformer_decoder1(*self.encode(src, c, tgt)))
        c = (x1 - src) ** 2
        x2 = self.fcn(self.transformer_decoder2(*self.encode(src, c, tgt)))
        return x1, x2

def train_tranad(model, Wtr, cfg):
    model.to(DEVICE)
    loader = DataLoader(TensorDataset(torch.tensor(Wtr, dtype=torch.float32)),
                        batch_size=cfg["BATCH"], shuffle=True, drop_last=True)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["LR"])
    mse = nn.MSELoss(reduction="none")
    epochs = cfg.get("TRANAD_EPOCHS", cfg["EPOCHS"])
    for ep in range(1, epochs+1):
        model.train(); n = ep; tot = 0.0
        for (wb,) in loader:
            wb = wb.to(DEVICE)                 # (B, W, F)
            src = wb.permute(1, 0, 2)          # (W, B, F)
            elem = src[-1:, :, :]              # (1, B, F) = last timestep
            x1, x2 = model(src, elem)
            loss = torch.mean((1/n)*mse(x1, elem) + (1 - 1/n)*mse(x2, elem))
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item()
        if ep % max(1, epochs//6) == 0:
            print(f"    epoch {ep:3d}/{epochs}  loss={tot/len(loader):.6f}")
    return model

@torch.no_grad()
def tranad_score(model, W, batch=256):
    model.eval(); out = []
    for i in range(0, len(W), batch):
        wb = torch.tensor(W[i:i+batch], dtype=torch.float32, device=DEVICE)
        src = wb.permute(1, 0, 2); elem = src[-1:, :, :]
        x1, x2 = model(src, elem)
        s = 0.5*((x1-elem)**2).mean(dim=2) + 0.5*((x2-elem)**2).mean(dim=2)  # (1, B)
        out.append(s.squeeze(0).cpu().numpy())
    return np.concatenate(out)

# ---------- GDN (Graph Deviation Network) — learns inter-sensor relations, SOTA on WADI ----------
# Compact faithful version of Deng & Hooi 2021. Each sensor is a graph node with a learned
# embedding; a top-k cosine graph over embeddings defines neighbours. The model FORECASTS the
# next value of every sensor from its window, aggregating neighbour features. Anomaly score =
# per-sensor forecast deviation, normalised per sensor, then max over sensors. Uses StandardScaler
# input (no Sigmoid), so it consumes the plain windows like LSTM-AE.
class GDN(nn.Module):
    def __init__(self, n_nodes, hist_len, embed_dim=64, hidden=64, topk=15):
        super().__init__()
        self.embedding = nn.Embedding(n_nodes, embed_dim)
        self.topk = min(topk, n_nodes - 1)
        self.lin = nn.Linear(hist_len, hidden)
        self.out = nn.Sequential(nn.Linear(hidden + embed_dim, hidden), nn.ReLU(),
                                 nn.Linear(hidden, 1))
    def _adj(self, device):
        e = torch.nn.functional.normalize(self.embedding.weight, dim=1)   # (N, E)
        sim = e @ e.t()                                                   # cosine (N, N)
        sim = sim - torch.eye(sim.size(0), device=device) * 2.0           # drop self
        topv, topi = sim.topk(self.topk, dim=1)
        A = torch.zeros_like(sim).scatter_(1, topi, torch.softmax(topv, dim=1))
        return A                                                          # row-normalised (N, N)
    def forward(self, x):                       # x: (B, N, hist_len)
        A = self._adj(x.device)
        g = torch.relu(self.lin(x))             # (B, N, H)
        agg = torch.einsum("ij,bjh->bih", A, g) # aggregate neighbour features (B, N, H)
        emb = self.embedding.weight.unsqueeze(0).expand(x.size(0), -1, -1)  # (B, N, E)
        z = torch.cat([agg, emb], dim=2)        # (B, N, H+E)
        return self.out(z).squeeze(-1)          # (B, N) = forecast next value per sensor

def train_gdn(model, Wtr, cfg):
    model.to(DEVICE)
    hist = torch.tensor(np.transpose(Wtr[:, :-1, :], (0, 2, 1)), dtype=torch.float32)  # (Nw, N, W-1)
    tgt  = torch.tensor(Wtr[:, -1, :], dtype=torch.float32)                             # (Nw, N)
    loader = DataLoader(TensorDataset(hist, tgt), batch_size=cfg["BATCH"], shuffle=True, drop_last=True)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["LR"]); mse = nn.MSELoss()
    epochs = cfg.get("GDN_EPOCHS", cfg["EPOCHS"])
    for ep in range(1, epochs+1):
        model.train(); tot = 0.0
        for xb, yb in loader:
            xb = xb.to(DEVICE); yb = yb.to(DEVICE)
            opt.zero_grad(); loss = mse(model(xb), yb)
            loss.backward(); opt.step(); tot += loss.item()
        if ep % max(1, epochs//6) == 0:
            print(f"    epoch {ep:3d}/{epochs}  loss={tot/len(loader):.5f}")
    return model

@torch.no_grad()
def gdn_errors(model, W, batch=256):
    model.eval(); out = []
    for i in range(0, len(W), batch):
        wb = W[i:i+batch]
        xb = torch.tensor(np.transpose(wb[:, :-1, :], (0, 2, 1)), dtype=torch.float32, device=DEVICE)
        tg = torch.tensor(wb[:, -1, :], dtype=torch.float32, device=DEVICE)
        out.append((model(xb) - tg).abs().cpu().numpy())
    return np.concatenate(out)                  # (num_windows, N) per-sensor forecast error

def gdn_score(err_val, err_test):
    """GDN deviation scoring: normalise each sensor's error by validation median/IQR, max over sensors."""
    med = np.median(err_val, axis=0)
    iqr = (np.percentile(err_val, 75, axis=0) - np.percentile(err_val, 25, axis=0)) + 1e-2
    sv = (np.abs(err_val  - med) / iqr).max(axis=1)
    st = (np.abs(err_test - med) / iqr).max(axis=1)
    return sv, st

## 7. Per-dataset runner

`run_all_models` performs the entire flow for one dataset (load → split → window → train every enabled model → evaluate) and returns the result rows plus the trained artifacts. Both the SWaT run and the WADI generalization run call this exact function — identical code, different data.

In [ ]:
def run_all_models(dataset_name, data_dir, cfg):
    # merge per-dataset overrides (e.g. longer WINDOW for WADI) and tag the dataset name
    cfg = {**cfg, **cfg.get("DATASET_OVERRIDES", {}).get(dataset_name, {}), "_dataset_name": dataset_name}
    print(f"\n===== {dataset_name} (WINDOW={cfg['WINDOW']}) =====")
    X, y = load_dataset(data_dir, cfg)
    Xtr, Xval, Xte, yte, scaler = make_splits(X, y, cfg)
    nf = Xtr.shape[1]
    W, S = cfg["WINDOW"], cfg["STRIDE"]
    Wtr, Wval, Wte = make_windows(Xtr,W,S), make_windows(Xval,W,S), make_windows(Xte,W,S)
    yte_win = window_labels(yte, W, S)

    results, models = [], {}
    win_scores = {}   # name -> (scores_val, scores_test) for window models, used by the ensemble

    if cfg["RUN"]["IsolationForest"]:
        t0=time.time()
        iso = IsolationForest(n_estimators=200, random_state=SEED, n_jobs=-1).fit(Xtr)
        results.append(evaluate("IsolationForest", dataset_name,
                        -iso.score_samples(Xval), -iso.score_samples(Xte), yte, cfg))
        print("    time:", round(time.time()-t0,1), "s")

    if cfg["RUN"]["DenseAE"]:
        t0=time.time()
        ae = train_ae(DenseAE(nf), Xtr, cfg)
        results.append(evaluate("DenseAE", dataset_name,
                        recon_error(ae, Xval, seq=False), recon_error(ae, Xte, seq=False), yte, cfg))
        models["DenseAE"]=ae; print("    time:", round(time.time()-t0,1), "s")

    if cfg["RUN"]["LSTM_AE"]:
        t0=time.time()
        lstm = train_ae(LSTMAE(nf), Wtr, cfg)
        sv, st = recon_error(lstm, Wval, seq=True), recon_error(lstm, Wte, seq=True)
        results.append(evaluate("LSTM_AE", dataset_name, sv, st, yte_win, cfg))
        win_scores["LSTM_AE"]=(sv, st); models["LSTM_AE"]=lstm; print("    time:", round(time.time()-t0,1), "s")

    if cfg["RUN"]["USAD"] or cfg["RUN"].get("TranAD"):
        mm_tf = fit_minmax_windows(Wtr)                 # shared [0,1] scaling for Sigmoid models
        Wtr_s, Wval_s, Wte_s = mm_tf(Wtr), mm_tf(Wval), mm_tf(Wte)

    if cfg["RUN"]["USAD"]:
        t0=time.time()
        usad = train_usad(USAD(W*nf), Wtr_s, cfg)
        sv, st = usad_score(usad, Wval_s), usad_score(usad, Wte_s)
        results.append(evaluate("USAD", dataset_name, sv, st, yte_win, cfg))
        win_scores["USAD"]=(sv, st); models["USAD"]=usad; print("    time:", round(time.time()-t0,1), "s")

    if cfg["RUN"].get("TranAD"):
        t0=time.time()
        tran = train_tranad(TranAD(nf, W), Wtr_s, cfg)
        sv, st = tranad_score(tran, Wval_s), tranad_score(tran, Wte_s)
        results.append(evaluate("TranAD", dataset_name, sv, st, yte_win, cfg))
        win_scores["TranAD"]=(sv, st); models["TranAD"]=tran; print("    time:", round(time.time()-t0,1), "s")

    if cfg["RUN"].get("GDN"):
        t0=time.time()
        gdn = train_gdn(GDN(nf, W-1, topk=cfg.get("GDN_TOPK", 15)), Wtr, cfg)
        sv, st = gdn_score(gdn_errors(gdn, Wval), gdn_errors(gdn, Wte))
        results.append(evaluate("GDN", dataset_name, sv, st, yte_win, cfg))
        win_scores["GDN"]=(sv, st); models["GDN"]=gdn; print("    time:", round(time.time()-t0,1), "s")

    # ---- Ensemble: z-score each member's scores and average (window models only) ----
    members = [m for m in cfg.get("ENSEMBLE_MEMBERS", []) if m in win_scores]
    if len(members) >= 2:
        def _z(a): return (a - a.mean()) / (a.std() + 1e-8)
        e_val  = np.mean([_z(win_scores[m][0]) for m in members], axis=0)
        e_test = np.mean([_z(win_scores[m][1]) for m in members], axis=0)
        results.append(evaluate("Ensemble(" + "+".join(members) + ")", dataset_name,
                        e_val, e_test, yte_win, cfg))

    return results, models, scaler, {"features": X.columns.tolist(), "n_features": int(nf)}

## 8. Run on SWaT (primary dataset)

In [ ]:
swat_results, swat_models, swat_scaler, swat_meta = run_all_models(
    "SWaT", CONFIG["DATASETS"]["SWaT"], CONFIG)

# Rank by PR-AUC: the honest, threshold-free metric. (The best-threshold "F1" column can
# read ~1.0 even for a broken model, so it must NOT be used to rank or to report.)
swat_df = pd.DataFrame(swat_results).sort_values("PR_AUC", ascending=False).reset_index(drop=True)
display(swat_df)

fig, ax = plt.subplots(figsize=(8,4))
swat_df.plot(x="Model", y=["ROC_AUC","PR_AUC","F1_fixed"], kind="bar", ax=ax)
ax.set_title("SWaT - honest metrics (ROC-AUC, PR-AUC, realistic F1)"); ax.set_ylim(0,1)
ax.set_ylabel("score"); plt.xticks(rotation=15); plt.legend(loc="lower right")
plt.tight_layout(); plt.show()
print("Best on SWaT (by PR-AUC):", swat_df.iloc[0]["Model"], "| PR-AUC =", swat_df.iloc[0]["PR_AUC"])

## 9. Save best artifacts (for deployment / inference)

Save the scaler and model weights so the inference service (Phase 4) can reuse them without retraining.

In [ ]:
import joblib
joblib.dump(swat_scaler, os.path.join(CONFIG["OUT_DIR"], "swat_scaler.pkl"))
json.dump({**swat_meta, "window": CONFIG["WINDOW"], "subsample": CONFIG["SUBSAMPLE"]},
          open(os.path.join(CONFIG["OUT_DIR"], "swat_meta.json"), "w"), indent=2)
for name, m in swat_models.items():
    torch.save(m.state_dict(), os.path.join(CONFIG["OUT_DIR"], f"swat_{name}.pt"))
swat_df.to_csv(os.path.join(CONFIG["OUT_DIR"], "swat_results.csv"), index=False)
print("Saved:", os.listdir(CONFIG["OUT_DIR"]))

## 10. Generalization — run WADI and compare with SWaT

**Key point:** the same model does **not** give the same numbers on a different plant — the weights learned SWaT's physics. What transfers is the whole pipeline. Here we run the identical `run_all_models` on **WADI** and stack the results next to SWaT in one table.

WADI is from the same lab (iTrust) and uses the same CSV format with a `Normal/Attack` column, so it loads with no code change. Just place its CSVs in `./data/wadi`. Expect F1 in a similar range but **lower and different** — WADI is larger and harder, which is normal and well-documented.

> Running one notebook across two datasets is concrete evidence for the "reusable reference architecture" claim.

In [ ]:
all_results = list(swat_results)   # start with SWaT rows

wadi_dir = CONFIG["DATASETS"].get("WADI")
if wadi_dir and glob.glob(os.path.join(wadi_dir, CONFIG["FILE_GLOB"])):
    wadi_results, wadi_models, wadi_scaler, wadi_meta = run_all_models("WADI", wadi_dir, CONFIG)
    all_results += wadi_results
    joblib.dump(wadi_scaler, os.path.join(CONFIG["OUT_DIR"], "wadi_scaler.pkl"))
    for name, m in wadi_models.items():
        torch.save(m.state_dict(), os.path.join(CONFIG["OUT_DIR"], f"wadi_{name}.pt"))
else:
    print("WADI folder not found or empty - showing SWaT only.")
    print("Put WADI CSVs in", wadi_dir, "and re-run this cell to compare.")

compare_df = pd.DataFrame(all_results)
compare_df.to_csv(os.path.join(CONFIG["OUT_DIR"], "comparison_all.csv"), index=False)
print("\n=== Combined comparison (SWaT vs WADI) ===")
display(compare_df.sort_values(["Model","Dataset"]).reset_index(drop=True))

### Combined F1 chart (SWaT vs WADI, per model)

In [ ]:
if compare_df["Dataset"].nunique() > 1:
    pivot = compare_df.pivot(index="Model", columns="Dataset", values="PR_AUC")
    fig, ax = plt.subplots(figsize=(8,4))
    pivot.plot(kind="bar", ax=ax)
    ax.set_title("PR-AUC per model: SWaT vs WADI"); ax.set_ylim(0,1)
    ax.set_ylabel("PR-AUC"); plt.xticks(rotation=15); plt.legend(title="Dataset")
    plt.tight_layout(); plt.savefig(os.path.join(CONFIG["OUT_DIR"], "swat_vs_wadi.png"), dpi=120)
    plt.show()
else:
    print("Add WADI data to see the side-by-side chart.")

## 10b. Robustness — multi-seed mean ± std (optional, heavy)

Re-runs the whole pipeline with several random seeds and reports the mean ± standard
deviation of PR-AUC / ROC-AUC per model, so you can show the results are stable and not a
lucky seed. This retrains every model N times — start with 3 seeds and reduce `RUN` /
raise `SUBSAMPLE` if it is too slow.

In [ ]:
RUN_ROBUSTNESS=False   # set True to run (heavy: retrains all models N times)
SEEDS=[42, 7, 123]
if RUN_ROBUSTNESS:
    import random as _rnd
    acc=[]
    for sd in SEEDS:
        _rnd.seed(sd); np.random.seed(sd); torch.manual_seed(sd)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(sd)
        r,_,_,_=run_all_models("SWaT", CONFIG["DATASETS"]["SWaT"], CONFIG)
        for row in r: acc.append({**row, "seed":sd})
    rob=pd.DataFrame(acc)
    agg=rob.groupby("Model")[["PR_AUC","ROC_AUC"]].agg(["mean","std"]).round(3)
    print("Multi-seed robustness (SWaT):"); display(agg)
    agg.to_csv(os.path.join(CONFIG["OUT_DIR"],"robustness_swat.csv"))
else:
    print("Set RUN_ROBUSTNESS=True to run the multi-seed stability check.")

## 11. Practical notes

- **Memory:** if the GPU runs out, lower `BATCH` or raise `SUBSAMPLE` (e.g. 10). USAD is heaviest because its input is `WINDOW x features`.
- **Time:** start with `EPOCHS: 5` and `SUBSAMPLE: 10` to verify the flow end-to-end, then scale up for the final run.
- **Threshold:** default `THRESHOLD_MODE="best_f1"` searches the threshold that maximizes F1 (fixes the F1=0-but-high-AUC rows). `F1_fixed` shows the old fixed-percentile result for comparison. Set the mode back to anything else to use the fixed percentile only.
- **USAD:** it uses its own MinMax [0,1] scaling + a Sigmoid output + gradient clipping, because its adversarial loss diverges (loss -> 1e23) when outputs are unbounded. Watch that L1 stays finite and decreases in the training log; if `ROC_AUC` is ~0.5 raise `USAD_EPOCHS`.
- **Report PR-AUC / ROC-AUC, not the best-threshold F1.** The `F1` column uses an oracle threshold searched on the test set with point-adjust, so it can read ~1.0 even for a broken model (that is exactly what happened to USAD before the fix). Rank and report by **PR-AUC** and **ROC-AUC**; use `F1_fixed` (threshold from validation) as the realistic deployment F1.
- **Point-adjust:** raises the numbers and is standard in SWaT papers, but state that you use it so comparisons stay fair.
- **Cross-dataset:** WADI/HAI will not match SWaT's exact F1 — retrain per plant and report each dataset's own numbers. WADI is genuinely harder; expect lower PR-AUC and plan real feature/model tuning, not just threshold fixes.
- **Deployment:** use `swat_scaler.pkl` + a `.pt` weights file in the inference service (Phase 4) with the exact same preprocessing before scoring.
